# Parallel Processing With Enhancements
Enhanced version accepts several functions:
* initializer - allows for a function to be called to setup initial conditions
* main - star of the show; the function to do the work; that which the other functions wrap around
* callback - a function to run once main has completed

### As for `concurrent.futures`
I investigated this functionality and found it good in ways, but too limited for what I wanted to do.  
* Futures can only be cancelled if the function called has not been started
* Callback function has no means of knowing the parameters supplied to the called function

### Imports

In [1]:
import concurrent.futures
import logging
import random
import threading
import time

## Setup Logging
Should functions emit messages to one or more of standard output, a log file, etc.?
Allows function output to be steered to files rather than rely on print statements

In [2]:
logging.basicConfig(level=logging.DEBUG,
					filename='parallel.log',
					filemode='w',
					format='%(asctime)s - %(levelname)s - %(message)s',
					datefmt='%Y-%m-%d %H:%M:%S', 
					force=True)

logger = logging.getLogger(__name__)

## Functions

### Helper functions

In [3]:
def info(a):
	return f'{a} [Type: {type(a)}]'

### Initializer

In [4]:
def my_init(fn, **kwargs):
    """
    Initialization function to be called immediately prior to invoking the main function.
    Records the start of execution, function called, and passed parameters
    
    fn: Function to be called
    **kwargs: dict of key / value paired parameters to pass to fn
    """
    logger.debug(f'Initializer for: {fn.__name__}({kwargs})')

### main
Various functions to call.  These do a little math and delay for a moment to illustrate threads finishing at different times.

In [5]:
def fn1(message: str, n: int, **kwargs):
    """
    A sample main function to be called.  
    This one sums the squares of a 1..n and returns the value.
    
    Parameters
    ----------------------------------------------------------------------------
    message: Function to be called

    **kwargs - dict of key / value paired parameters to pass to fn

    Returns
    ----------------------------------------------------------------------------
    None
    """
    
    # Calculate the return value
    ret = 0
    for i in range(1, n + 1):
        ret += i**2

    # Wait a moment
    r = n * random.random()
    time.sleep(r)

    # Print a message and return the value
    logger.debug(f'Done in {r:.2f} seconds -- (random wait factor {n})' if n > 1 else '')
    return ret


def fn2(**kwargs):
    """
    A sample main function to be called.  
    This one returns the n!, if given; otherwise returns 1
    
    message: Function to be called
    **kwargs - dict of key / value paired parameters to pass to fn
    """

    # Catch the given value for n, use 1 if no such parameter
    n = kwargs['n'] if kwargs.__contains__('n') else 1
    
    # Calculate the return value
    ret = 1
    for i in range(1, n + 1):
        ret *= i

    # Wait a moment    
    r = n * random.random()
    time.sleep(r)

    # Print a message and return the value
    logger.debug(f'Done in {r:.2f} seconds -- (random wait factor {n})' if kwargs.__contains__('n') else '')
    return ret

### Callback

In [6]:
def my_callback(fn, **kwargs):
    logger.debug(f'Callback for: {fn.__name__}({kwargs})')

### Parallelize
Accepts a list of functions and the keyword arguments to pass them.  
Invokes a configurable number of these in parallel, invoking the next when a thread becomes available,
eventually calling all in the list.

In [16]:
def null_fn(fn, **kwargs):
    """
    A simple function that always passes; intended for use as the default when
    a function can be passed as an optional argument in another function call    
    """
    pass

def parallelize(function_calls: list = [], init = null_fn, callback = null_fn, maxDoP: int = 8):
    """
    Executes the list of functions provided in parallel.
    Optionally calling initializer & callback functions around the execution of each.

    Parameters
    ----------------------------------------------------------------------------------------------
    function_calls: List of tuples (function, **kwargs) to execute

    init:           Optional initializer function to call prior to execution of each function
                    Must have (fn, **kwargs) signature

    callback:       Optional callback function to call upon completion of each function
                    Must have (fn, **kwargs) signature

    maxDoP:         Degree if parallelism - limits number of functions to run simultaneously

    Returns
    ----------------------------------------------------------------------------------------------
    None
    """

    # Setup threads and work to perform
    threads = set()
    startedThreads = 0
    todo = function_calls.copy()
    
    with concurrent.futures.ThreadPoolExecutor() as tpExec:
        # While tasks remain to be done...
        while len(todo) > 0:
            # ... and we're not already running maxDoP
            while len(threads) < maxDoP:
                # If we've run out of things to do, break from the loop
                if len(todo) == 0:
                    break

                # Get the next function and its keyword arguments and setup a thread to run it
                fn, kwargs = todo.pop()
                init(fn, **kwargs)

                # Add the thread to our running list and start it
                threads.add(tpExec.submit(fn(**kwargs)))
                startedThreads += 1

            # Wait for a thread to finish
            while len(threads) == maxDoP:
                for rt in threads:
                    if not rt.done():
                        startedThreads -= 1
                        print(rt.result())


    # for fn, kwargs in function_calls:
    #     if init is not None:
    #         init(fn, **kwargs)

    #         try:
    #             logger.debug(f'Result: {fn(**kwargs)}')

    #         except Exception as ex:
    #             logger.error(ex)

    #     if callback is not None:
    #         callback(fn, **kwargs)

In [17]:
call_sheet = [
	(fn1, {'message':'Summing squares through 4 should be 30...', 'n':4, 'r':30}),
	(fn1, {'message':'Summing squares through 7 should be 140', 'n':7, 'r':140}),
#	(fn1, {'message':"Oh, no! I'm gonna fail on missing parameter 'n'", 'r':0}),
#	(fn1, {'message':"Oh, no! I'm gonna fail due to negative 'n'", 'n': -2, 'r':0}),
	(fn2, {'dummy':'no delay'}),
	(fn2, {'dummy':'Delay specified', 'n': 7, 'r': 111}),
	]

parallelize(call_sheet, init=my_init, callback=my_callback, maxDoP=3)

TypeError: 'int' object is not callable